# **Video Processing Week 3: Analisis Tangan & Aplikasi Interaktif**

Minggu ini kita akan mempelajari **analisis tangan dan aplikasi interaktif** menggunakan teknologi AI melalui MediaPipe. Fokus utama adalah deteksi dan tracking tangan serta jari untuk membangun aplikasi interaktif yang dapat merespons gestur tangan secara real-time.

**Tujuan Pembelajaran:**

Di akhir sesi ini kita akan mampu:
- Menggunakan MediaPipe Hand untuk mendeteksi dan melacak tangan secara real-time
- Mengimplementasikan sistem pengenalan gestur sederhana berdasarkan posisi landmark
- Membangun aplikasi penghitung jari dan deteksi gestur dasar
- Mengembangkan proyek Virtual Painter menggunakan gestur tangan sebagai kontrol

**Topik Praktik:**
- **Hand Landmark Detection**: Deteksi dan tracking 21 titik landmark pada tangan
- **Hand Gesture Recognition**: Pengenalan gestur sederhana seperti menghitung jari dan membedakan kepalan vs telapak terbuka
- **Virtual Painter Project**: Aplikasi interaktif untuk menggambar menggunakan gestur jari telunjuk dan mengubah warna/menghapus dengan gestur telapak terbuka

> *This module is inspired by the development of last semester’s materials.* 


*Sebelum lanjut, kita coba import library yang akan kita butuhkan dulu*

In [ ]:
# pip install opencv-contrib-python numpy matplotlib mediapipe ipykernel
# atau
# uv pip install opencv-contrib-python numpy matplotlib mediapipe ipykernel

In [1]:
import cv2
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt
import os

## Hand Landmark Detection

kita akan melakukan deteksi 21 titik landmark pada satu atau dua tangan melalui webcam secara real-time

In [5]:
# Inisialisasi MediaPipe Hands
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

# Konfigurasi Hand Detection
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=2, min_detection_confidence=0.5, min_tracking_confidence=0.5)

# Buka webcam dan siapkan canvas/variabel
cap = cv2.VideoCapture(0)
canvas = np.zeros((480, 640, 3), dtype=np.uint8)
prev_x, prev_y = None, None

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: 
        break

    # Membalikkan frame secara horizontal agar tidak mirror
    frame = cv2.flip(frame, 1)

    # Sesuaikan ukuran canvas dengan webcam
    if canvas.shape[:2] != frame.shape[:2]:
        canvas = np.zeros_like(frame)

    # Konversi BGR ke RGB & proses deteksi
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)

    # Gambar landmark jika tangan terdeteksi
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            # Gambar landmark dan koneksi
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS, mp_drawing_styles.get_default_hand_landmarks_style(), mp_drawing_styles.get_default_hand_connections_style())

            # Ambil landmark ujung telunjuk (8) dan sendi PIP telunjuk (6)
            index_tip = hand_landmarks.landmark[8]
            index_pip = hand_landmarks.landmark[6]

            # Konversi posisi ujung telunjuk ke piksel layar
            x, y = int(index_tip.x * frame.shape[1]), int(index_tip.y * frame.shape[0])

            # JIKA TELUNJUK DIANGKAT (Nilai Y Tip < Y Pip)
            if index_tip.y < index_pip.y:
                # Pena virtual (indikator lingkaran)
                cv2.circle(frame, (x, y), 10, (255, 0, 255), -1)

                # Menggambar garis jika koordinat sebelumnya ada
                if prev_x is not None and prev_y is not None:
                    cv2.line(canvas, (prev_x, prev_y), (x, y), (255, 0, 255), 5)

                prev_x, prev_y = x, y
            else:
                # JIKA TELUNJUK DITUTUP: Putus jalur coretan agar tidak menyambung saat diangkat lagi
                prev_x, prev_y = None, None
    else:
        prev_x, prev_y = None, None

    # Gabungkan canvas dan webcam
    frame = cv2.add(frame, canvas)

    # Tambahkan teks instruksi
    cv2.putText(frame, "Virtual Pen", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 255), 2)
    cv2.putText(frame, "C = Clear | Q = Quit", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    cv2.imshow("Hand Landmark Detection", frame)
    key = cv2.waitKey(1) & 0xFF

    if key == ord('c'): canvas[:] = 0
    if key == ord('q'): break

cap.release()
cv2.destroyAllWindows()
hands.close()

## Hand Gesture Recognition Sederhana

Kita akan mengimplementasikan pengenalan gestur sederhana dengan menerjemahkan data landmark menjadi informasi gestur seperti menghitung jari yang terangkat dan membedakan antara kepalan tangan dengan telapak terbuka.

### Menerjemahkan data landmark menjadi gestur

Berdasarkan kode sebelumnya, MediaPipe hanya memberi Anda 21 titik data mentah. Ia tidak tahu apa arti "menunjuk" atau "mengepal". Anda harus memberitahunya menggunakan logika Python.

*Lalu bagaimana caranya?* Simpel, dengan cara mengukur jarak atau membandingkan posisi antar titik.

#### **Contoh A: Gestur "Menunjuk" ☝️**

**Logika:** "Sebuah tangan dianggap 'menunjuk' JIKA..."
- Ujung jari telunjuk (Titik 8) lurus dan jauh dari telapak tangan
- **DAN** ujung jari tengah (Titik 12), jari manis (Titik 16), dan kelingking (Titik 20) posisinya "melipat" atau dekat dengan telapak tangan

#### **Contoh B: Gestur "Tangan Mengepal" ✊**

**Logika:** "Sebuah tangan dianggap 'mengepal' JIKA..."
- Semua ujung jari (Titik #8, #12, #16, #20) posisinya dekat dengan telapak tangan
- Jempol (Titik #4) juga dalam posisi melipat

#### Kesimpulan

Dengan membandingkan posisi relatif titik-titik landmark tangan, Anda dapat mengartikan gestur tangan tertentu. Logika ini dapat diperluas untuk mengenali gestur yang lebih kompleks sesuai kebutuhan aplikasi kita.

### Praktik: Menghitung jumlah jari yang terangkat

Untuk menghitung jumlah jari yang terangkat, kita bisa melakukan deteksi ujung dan dasar dari masing-masing jari kemudian membandingkan titiknya secara vertikal:

- jika ujung jari lebih tinggi dari dasar/tengah jarinya, maka jari tersebut terangkat. Secara programatik: `finger_tip.y < finger_base.y`

*Catatan: nilai y=0 dimulai dari atas frame. Artinya, nilai y yang lebih kecil menandakan titik ada di bagian atas dan nilai y yang lebih besar ada di bagian bawah*

In [8]:
import cv2
import mediapipe as mp

def count_fingers(hand_landmarks, hand_label):
    # Jari selain jempol: [TIP_ID, PIP_ID]
    finger_tips = [
        [8, 6],   # Telunjuk
        [12, 10], # Tengah
        [16, 14], # Manis
        [20, 18]  # Kelingking
    ]
    fingers_up = 0
    
    # Logika untuk 4 Jari (Sumbu Y)
    for tip_id, pip_id in finger_tips:
        if hand_landmarks.landmark[tip_id].y < hand_landmarks.landmark[pip_id].y:
            fingers_up += 1
            
    # Logika Khusus Jempol (Sumbu X)
    thumb_tip = hand_landmarks.landmark[4]
    thumb_mcp = hand_landmarks.landmark[2]
    
    if hand_label == "Right":
        if thumb_tip.x < thumb_mcp.x: fingers_up += 1
    else:
        if thumb_tip.x > thumb_mcp.x: fingers_up += 1
            
    return fingers_up

# Inisialisasi MediaPipe
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    frame = cv2.flip(frame, 1)
    h, w = frame.shape[:2]
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)
    
    left_fingers = 0
    right_fingers = 0
    
    if results.multi_hand_landmarks and results.multi_handedness:
        for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
            # Gambar landmark tangan
            mp_drawing.draw_landmarks(
                frame, hand_landmarks, mp_hands.HAND_CONNECTIONS,
                mp_drawing_styles.get_default_hand_landmarks_style(),
                mp_drawing_styles.get_default_hand_connections_style()
            )
            
            hand_label = handedness.classification[0].label
            finger_count = count_fingers(hand_landmarks, hand_label)
            
            if hand_label == "Left":
                left_fingers = finger_count
            elif hand_label == "Right":
                right_fingers = finger_count

    total = left_fingers + right_fingers
    
    # --- BAGIAN TAMPILAN (UI) YANG SUDAH DIPERKECIL & DIRAPATKAN ---
    # 1. Judul Aplikasi (Kecil & Tipis)
    cv2.putText(frame, "Kalkulator Jari Virtual", (20, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
    # 2. Status Jari Kiri & Kanan
    cv2.putText(frame, f"Kiri: {left_fingers} | Kanan: {right_fingers}", (20, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 1)
    
    # 3. Hasil Penjumlahan (Dibuat pas, tidak terlalu raksasa)
    cv2.putText(frame, f"Hasil: {left_fingers} + {right_fingers} = {total}", (20, 85), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    
    # 4. Instruksi Keluar di pojok bawah
    cv2.putText(frame, "Tekan 'q' untuk keluar", (20, h - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
    
    cv2.imshow("Kalkulator Jari", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()
hands.close()

#### 🔎 Eksplorasi

- Menggunakan logika sederhana yang sudah kita implementasikan, meskipun kita telah mengepalkan tangan kita, program masih membaca ada satu jari yang terangkat. Jari apakah yang menyebabkan hal tersebut? dan ide apa yang kamu punya untuk mengatasi masalah ini?

*Hint: Jempol memiliki orientasi berbeda, mungkin perlu logika terpisah*

### Praktik: Membedakan gestur kepalan tangan VS telapak terbuka

Untuk membedakan gestur kepalan tangan vs telapak terbuka, kita dapat menganalisis posisi landmark jari-jari relatif terhadap telapak tangan. Berikut adalah tahapan implementasinya:

1. Analisis Posisi Jari
Kita membandingkan posisi ujung jari (tip) dengan sendi tengah (PIP) atau pangkal jari:
- **Jari terangkat**: Ujung jari berada di atas sendi tengah (koordinat y lebih kecil)
- **Jari tertutup**: Ujung jari berada di bawah atau sejajar dengan sendi tengah

2. Logika Penentuan Gestur

**Kepalan Tangan (Fist):**
- Semua ujung jari (index 8, 12, 16, 20) berada di bawah sendi tengahnya
- Jempol (index 4) tertutup ke dalam telapak tangan
- Kondisi: `finger_count == 0` atau semua jari dalam posisi tertutup

**Telapak Terbuka (Open Palm):**
- Semua ujung jari berada di atas sendi tengahnya
- Jempol terangkat dan terpisah dari telapak tangan
- Kondisi: `finger_count >= 4` dan jarak antar jari cukup lebar, atau `tip.y < base.y`

In [11]:
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

def detect_gesture(hand_landmarks):
    # Cek status terangkat (True/False) untuk masing-masing dari 4 jari
    index_up = hand_landmarks.landmark[8].y < hand_landmarks.landmark[6].y
    middle_up = hand_landmarks.landmark[12].y < hand_landmarks.landmark[10].y
    ring_up = hand_landmarks.landmark[16].y < hand_landmarks.landmark[14].y
    pinky_up = hand_landmarks.landmark[20].y < hand_landmarks.landmark[18].y
    
    # 1. Logika GUNTING (Telunjuk & Tengah naik, Manis & Kelingking turun)
    if index_up and middle_up and not ring_up and not pinky_up:
        return "GUNTING", (255, 0, 0) # Warna Biru
        
    # 2. Logika KERTAS / OPEN PALM (Minimal 3 jari terangkat)
    elif (index_up + middle_up + ring_up + pinky_up) >= 3:
        return "KERTAS", (0, 255, 0) # Warna Hijau
        
    # 3. Logika BATU / FIST (Jika tidak memenuhi kondisi di atas)
    else:
        return "BATU", (0, 0, 255) # Warna Merah

hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7, min_tracking_confidence=0.7)
cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    frame = cv2.flip(frame, 1)
    h, w = frame.shape[:2]
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)
    
    # Default status jika tidak ada tangan terdeteksi
    gesture_text, color = "NO HAND", (255, 255, 255)
    
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            # Panggil fungsi deteksi gestur baru
            gesture_text, color = detect_gesture(hand_landmarks)
            
    # Tampilkan UI hasil deteksi
    cv2.putText(frame, f"Status: {gesture_text}", (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
    cv2.putText(frame, "Press 'q' to quit", (10, h - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    
    cv2.imshow("Rock Paper Scissors Detector", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()
hands.close()

## Proyek Mini: Virtual Painter

Kita akan membuat sebuah aplikasi Virtual Painter yang memungkinkan pengguna menggambar di layar menggunakan gestur tangan:

- Menggabungkan deteksi tangan dan gestur
- Gestur jari telunjuk terangkat -> menggambar
- Gestur telapak tangan terbuka -> mengubah warna
- Gestur tangan ditutup -> menghapus gambar

In [14]:
def hand_drawing():
    cap = cv2.VideoCapture(0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    mp_hands = mp.solutions.hands
    hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.5, min_tracking_confidence=0.5)
    mp_drawing = mp.solutions.drawing_utils
    
    # BARU: Pengaturan Warna Default & Ketebalan
    drawing_color = (0, 0, 255)  # Default: Merah (BGR)
    thickness = 5
    canvas = np.zeros((height, width, 3), dtype=np.uint8)
    prev_finger_pos = None
    
    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return
        
    while True:
        ret, frame = cap.read()
        if not ret: break
            
        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb_frame)
        
        # BARU: Gambar Kotak Menu Pilihan Warna di Atas Layar (Header)
        # Pembagian 4 Kotak: Merah, Hijau, Biru, Penghapus
        box_w = width // 4
        cv2.rectangle(frame, (0, 0), (box_w, 60), (0, 0, 255), -1)       # Kotak Merah
        cv2.rectangle(frame, (box_w, 0), (box_w*2, 60), (0, 255, 0), -1)  # Kotak Hijau
        cv2.rectangle(frame, (box_w*2, 0), (box_w*3, 60), (255, 0, 0), -1)# Kotak Biru
        cv2.rectangle(frame, (box_w*3, 0), (width, 60), (200, 200, 200), -1) # Kotak Penghapus
        
        # Teks Label Menu
        cv2.putText(frame, "RED", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(frame, "GREEN", (box_w + 20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(frame, "BLUE", (box_w*2 + 20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(frame, "ERASER", (box_w*3 + 20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                
                # Koordinat Ujung Jari
                x = int(hand_landmarks.landmark[8].x * width)
                y = int(hand_landmarks.landmark[8].y * height)
                
                # Deteksi Jari Terangkat (Bandingkan Tip dengan PIP/Pangkal)
                index_up = hand_landmarks.landmark[8].y < hand_landmarks.landmark[6].y
                middle_up = hand_landmarks.landmark[12].y < hand_landmarks.landmark[10].y
                ring_up = hand_landmarks.landmark[16].y < hand_landmarks.landmark[14].y
                pinky_up = hand_landmarks.landmark[20].y < hand_landmarks.landmark[18].y
                
                is_open_palm = index_up and middle_up and ring_up and pinky_up
                
                if is_open_palm:
                    # Gestur 5 Jari: Hapus total kanvas
                    canvas = np.zeros((height, width, 3), dtype=np.uint8)
                    prev_finger_pos = None
                    
                elif index_up and middle_up:
                    # BARU: GESTUR 2 JARI (Telunjuk + Tengah) = MODE SELEKSI WARNA
                    prev_finger_pos = None # Putus jalur coretan agar tidak ikut menggambar
                    cv2.circle(frame, (x, y), 10, (255, 255, 255), -1) # Indikator lingkaran putih
                    
                    # Cek jika ujung jari berada di dalam area menu atas (y < 60)
                    if y < 60:
                        if 0 < x < box_w:
                            drawing_color = (0, 0, 255) # Pilih Merah
                            thickness = 5
                        elif box_w < x < box_w*2:
                            drawing_color = (0, 255, 0) # Pilih Hijau
                            thickness = 5
                        elif box_w*2 < x < box_w*3:
                            drawing_color = (255, 0, 0) # Pilih Biru
                            thickness = 5
                        elif box_w*3 < x < width:
                            drawing_color = (0, 0, 0)   # Mode Penghapus (Menggambar dengan warna hitam)
                            thickness = 50              # Ukuran penghapus dibuat lebih besar
                            
                elif index_up:
                    # GESTUR 1 JARI (Telunjuk Saja) = MODE MENGGAMBAR
                    # Indikator warna lingkaran mengikuti warna aktif
                    cv2.circle(frame, (x, y), thickness // 2 + 5, drawing_color, -1)
                    
                    if prev_finger_pos is not None:
                        cv2.line(canvas, prev_finger_pos, (x, y), drawing_color, thickness)
                    prev_finger_pos = (x, y)
                else:
                    prev_finger_pos = None
        else:
            prev_finger_pos = None
            
        # Gabungkan frame asli dengan kanvas gambar
        combined_image = cv2.addWeighted(frame, 0.7, canvas, 0.7, 0)
        
        # Tampilkan Status Warna Aktif
        status_text = "ERASER" if drawing_color == (0,0,0) else "DRAWING"
        cv2.putText(combined_image, f"Mode: {status_text}", (10, height - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        
        cv2.imshow('Air MS Paint App', combined_image)
        
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'): break
        elif key == ord('c'):
            canvas = np.zeros((height, width, 3), dtype=np.uint8)
            prev_finger_pos = None
            
    cap.release()
    cv2.destroyAllWindows()

hand_drawing()

Dari proyek mini ini, ada beberapa topik penting dalam mencapai fungsionalitas yang kita inginkan:

### Menggabungkan deteksi tangan dan gestur

Tahapan:
- Landmark tangan di frame saat ini dideteksi menggunakan mediapipe dengan sintaks `results = hands.process(rgb_frame)`
- Dapatkan posisi dari ujung dan dasar jari telunjuk
- Jika ujung jari lebih tinggi dari dasar jari, maka buat garis dari titik sebelumnya: `cv2.line(canvas, prev_finger_pos, (x, y), drawing_color, drawing_thickness)`
- Simpan posisi jari sekarang sebagai referensi untuk frame berikutnya
- sebaliknya, Jika ujung jari lebih rendah dari dasar jari, hentikan penggambaran garis dan skip frame saat ini `else: prev_finger_pos = None`


### Menggunakan gestur (misal: jari telunjuk terangkat) untuk menggambar di layar.

- Jika ujung jari lebih tinggi dari dasar jari, maka buat garis dari titik sebelumnya: `cv2.line(canvas, prev_finger_pos, (x, y), drawing_color, drawing_thickness)`
- Simpan posisi jari sekarang sebagai referensi untuk frame berikutnya
- sebaliknya, Jika ujung jari lebih rendah dari dasar jari, hentikan penggambaran garis dan skip frame saat ini `else: prev_finger_pos = None`

### Menggunakan gestur lain (telapak terbuka) untuk menghapus kanvas.

Tahapan deteksi gestur telapak terbuka:
- Periksa status semua jari (telunjuk, tengah, manis, kelingking) dengan membandingkan posisi y ujung vs pangkal jari
- Jari dianggap "terangkat" jika koordinat y ujung lebih kecil dari y pangkal: `index_up = y < base_y`
- Untuk jari lainnya: `middle_up = hand_landmarks.landmark[12].y < hand_landmarks.landmark[9].y`
- Jika semua jari terangkat (`is_open_palm = index_up and middle_up and ring_up and pinky_up`), hapus kanvas
- Reset kanvas dengan membuat array kosong: `canvas = np.zeros((height, width, 3), dtype=np.uint8)`
- Reset posisi jari sebelumnya: `prev_finger_pos = None`

## Kesimpulan

Dalam modul **Video Processing Week 3: Analisis Tangan & Aplikasi Interaktif**, kita telah mempelajari implementasi teknologi AI untuk deteksi dan pengenalan gestur tangan menggunakan MediaPipe. Berikut adalah rangkuman pencapaian pembelajaran:

### 🎯 Pencapaian Utama

**1. Hand Landmark Detection**
- Berhasil mengimplementasikan deteksi 21 titik landmark pada tangan secara real-time
- Memahami struktur data landmark MediaPipe dan cara visualisasinya
- Mengonfigurasi parameter deteksi untuk optimalisasi akurasi dan performa

**2. Hand Gesture Recognition**
- Mengembangkan logika penerjemahan data landmark mentah menjadi informasi gestur
- Implementasi penghitung jari dengan membandingkan posisi relatif ujung dan pangkal jari
- Membedakan gestur kepalan tangan vs telapak terbuka menggunakan analisis posisi landmark

**3. Virtual Painter Application**
- Membangun aplikasi interaktif yang merespons gestur tangan secara real-time
- Mengintegrasikan multiple gesture recognition: menggambar (jari telunjuk), menghapus (telapak terbuka)
- Menerapkan teknik overlay canvas untuk visualisasi hasil gambar

### 🔑 Konsep Kunci yang Dipelajari

- **Coordinate System**: Memahami sistem koordinat MediaPipe (nilai y=0 di atas frame)
- **Landmark Analysis**: Teknik membandingkan posisi relatif antar landmark untuk interpretasi gestur
- **Real-time Processing**: Implementasi pipeline deteksi dan response dalam aplikasi interaktif
- **Computer Vision Integration**: Menggabungkan OpenCV dan MediaPipe untuk solusi vision yang kompleks

### 💡 Aplikasi Praktis

Materi ini memberikan foundation yang kuat untuk pengembangan:
- Aplikasi kontrol gestur untuk presentasi atau gaming
- Sistem antarmuka touchless untuk lingkungan steril
- Aplikasi edukasi interaktif untuk anak-anak
- Prototyping human-computer interaction yang inovatif

### 🚀 Pengembangan Selanjutnya

Dengan pemahaman dasar ini, pembelajaran dapat dilanjutkan ke:
- Gesture recognition yang lebih kompleks (sign language, custom gestures)
- Integration dengan machine learning untuk pattern recognition
- Multi-modal interaction (kombinasi hand tracking dengan voice/eye tracking)
- Optimalisasi performa untuk deployment pada edge devices